In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [2]:
# Подготовка данных (пример)
X = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_road_2_1.csv')
S = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_speed_2_1.csv')
y = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\y_train_wheel_2_1.csv')

In [19]:
# Подготовка данных (пример)
X = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_road_2_2.csv')
S = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_speed_2_2.csv')
y = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\y_train_wheel_2_2.csv')

In [20]:
X_train = torch.tensor(X.values, dtype=torch.float32).to(device='cuda')
S_train = torch.tensor(S.values, dtype=torch.float32).to(device='cuda')
y_tensor = torch.tensor(y.values, dtype=torch.float32).to(device='cuda')

In [21]:
print(len(X_train))
print(len(S_train))
print(len(y_tensor))

17050
17050
17050


In [22]:
X_tensor = torch.cat((X_train, S_train), dim=1)
print(X_tensor[0])
print(X_tensor[0].size())

tensor([ 0.,  0.,  0.,  ...,  0.,  0., 35.], device='cuda:0')
torch.Size([12289])


In [23]:
print(y_tensor[0])
print(y_tensor[0].size())

tensor([-0.0029, -0.0029], device='cuda:0')
torch.Size([2])


In [24]:
dataset = TensorDataset(X_tensor, y_tensor)
data_loader = DataLoader(dataset, batch_size=512, shuffle=False)

In [8]:
class FeedforwardNet(nn.Module):
    def __init__(self, input_size=12289, hidden_size_1=512, hidden_size_2=256, hidden_size_3=128,
                 output_size=2):
        super(FeedforwardNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size_1)
        self.fc2 = nn.Linear(hidden_size_1, hidden_size_2)
        self.fc3 = nn.Linear(hidden_size_2, hidden_size_3)
        self.fc4 = nn.Linear(hidden_size_3, output_size)
        self.relu = nn.ReLU()
        self.tanh = nn.Tanh()

    def forward(self, input_data):
        out = self.fc1(input_data)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        out = self.relu(out)
        out = self.fc4(out)
        out = self.tanh(out)
        return out

In [17]:
model = FeedforwardNet().cuda()

In [25]:
# Определение функции потерь и оптимизатора
criterion = nn.MSELoss()  # Функция потерь для регрессионной задачи
optimizer = optim.Adam(model.parameters())  # Оптимизатор Adam

# Обучение модели
num_epochs = 2
for epoch in range(num_epochs):
    for batch_x, batch_y in data_loader:  # Итерация по батчам данных
        optimizer.zero_grad()  # Обнуление градиентов

        outputs = model(batch_x.unsqueeze(0))  # Передача входных данных через модель
        loss = criterion(outputs, batch_y)  # Вычисление потерь

        loss.backward()  # Обратное распространение ошибки
        optimizer.step()  # Обновление весов модели

    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.9f}')

C:\PycharmProjects\ETS_Autopilot\venv_3_11\Lib\site-packages\torch\nn\modules\loss.py:535: UserWarning: Using a target size (torch.Size([512, 2])) that is different to the input size (torch.Size([1, 512, 2])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch [1/2], Loss: 0.000099313
Epoch [2/2], Loss: 0.000092932


C:\PycharmProjects\ETS_Autopilot\venv_3_11\Lib\site-packages\torch\nn\modules\loss.py:535: UserWarning: Using a target size (torch.Size([154, 2])) that is different to the input size (torch.Size([1, 154, 2])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


In [26]:
torch.save(model.state_dict(), 'C:\PycharmProjects\ETS_Autopilot\static\weight_model\weight_wheel_nn_forward_4.pth')